# Visualization: Plotting Knowledge Values & Uncertainty Bands

This notebook demonstrates visualization of epistemic data:
- Bar charts with uncertainty error bars
- Concentration-time profiles with uncertainty bands
- Customizing plots for publication

In [ ]:
import sys
sys.path.insert(0, '../sounio-py/python')

from sounio.knowledge import Knowledge

# Try to import matplotlib, continue if not available
try:
    import matplotlib.pyplot as plt
    import numpy as np
    HAS_MATPLOTLIB = True
except ImportError:
    HAS_MATPLOTLIB = False
    print("NOTE: matplotlib not installed. Code examples shown but cannot execute.")
    print("Install with: pip install matplotlib numpy")

if HAS_MATPLOTLIB:
    print("matplotlib and numpy loaded successfully")

## Part 1: Bar Charts with Error Bars

In [ ]:
# Example 1: Pharmacokinetic Parameters
if HAS_MATPLOTLIB:
    # Create Knowledge values
    pk_params = {
        "Half-life (h)": Knowledge(4.62, 0.767, "pk_fit"),
        "Clearance (L/h)": Knowledge(12.5, 1.5, "pk_fit"),
        "V_d (L)": Knowledge(85.0, 5.0, "pk_fit"),
    }
    
    # Extract values and uncertainties
    names = list(pk_params.keys())
    values = [pk_params[name].value for name in names]
    errors = [pk_params[name].epsilon for name in names]
    
    # Normalize for display (half-life in different scale)
    fig, ax = plt.subplots(figsize=(10, 6))
    x = np.arange(len(names))
    ax.bar(x, values, yerr=errors, capsize=10, alpha=0.7, color='steelblue', edgecolor='black')
    ax.set_ylabel('Parameter Value', fontsize=12)
    ax.set_title('Pharmacokinetic Parameters with Uncertainty', fontsize=14, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=45, ha='right')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("# Code to create bar chart:")
    print("""import matplotlib.pyplot as plt
import numpy as np

pk_params = {
    'Half-life (h)': Knowledge(4.62, 0.767, 'pk_fit'),
    'Clearance (L/h)': Knowledge(12.5, 1.5, 'pk_fit'),
    'V_d (L)': Knowledge(85.0, 5.0, 'pk_fit'),
}

names = list(pk_params.keys())
values = [pk_params[name].value for name in names]
errors = [pk_params[name].epsilon for name in names]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(names))
ax.bar(x, values, yerr=errors, capsize=10, alpha=0.7, color='steelblue', edgecolor='black')
ax.set_ylabel('Parameter Value', fontsize=12)
ax.set_title('Pharmacokinetic Parameters with Uncertainty', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=45, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()""")

## Part 2: Concentration-Time Profile with Uncertainty Band

In [ ]:
if HAS_MATPLOTLIB:
    # Simulated concentration-time profile
    # C(t) = (Dose / Vd) * exp(-k * t), where k = ln(2) / t_half
    
    dose = Knowledge(500.0, 10.0, "weighing_scale")  # 500 mg ± 10 mg
    v_d = Knowledge(80.0, 4.0, "fitting")             # 80 L ± 4 L
    t_half = Knowledge(4.5, 0.5, "fitting")           # 4.5 h ± 0.5 h
    
    # Time points (hours)
    times = np.array([0, 0.5, 1.0, 2.0, 4.0, 8.0, 12.0, 24.0, 48.0])
    
    # Calculate central estimate and bounds
    k_central = 0.693 / t_half.value
    c_central = (dose.value / v_d.value) * np.exp(-k_central.value * times)
    
    # Approximate 95% confidence interval (±2σ) by propagating uncertainty
    # This is a simplified approach; proper uncertainty propagation shown below
    c_upper = c_central * (1 + 0.20)  # Rough 20% higher bound
    c_lower = c_central * (1 - 0.20)  # Rough 20% lower bound
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(times, c_central, 'o-', linewidth=2, markersize=6, label='Central estimate', color='navy')
    ax.fill_between(times, c_lower, c_upper, alpha=0.3, color='lightblue', label='95% confidence interval')
    ax.set_xlabel('Time (h)', fontsize=12)
    ax.set_ylabel('Concentration (mg/L)', fontsize=12)
    ax.set_title('Pharmacokinetic Profile with Uncertainty Band', fontsize=14, fontweight='bold')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print("# Code to create concentration-time profile:")
    print("""import matplotlib.pyplot as plt
import numpy as np

dose = Knowledge(500.0, 10.0, 'weighing_scale')
v_d = Knowledge(80.0, 4.0, 'fitting')
t_half = Knowledge(4.5, 0.5, 'fitting')

times = np.array([0, 0.5, 1.0, 2.0, 4.0, 8.0, 12.0, 24.0, 48.0])

k_central = 0.693 / t_half.value
c_central = (dose.value / v_d.value) * np.exp(-k_central * times)

c_upper = c_central * (1 + 0.20)
c_lower = c_central * (1 - 0.20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(times, c_central, 'o-', linewidth=2, markersize=6, label='Central estimate', color='navy')
ax.fill_between(times, c_lower, c_upper, alpha=0.3, color='lightblue', label='95% confidence interval')
ax.set_xlabel('Time (h)', fontsize=12)
ax.set_ylabel('Concentration (mg/L)', fontsize=12)
ax.set_title('Pharmacokinetic Profile with Uncertainty Band', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()""")

## Part 3: Relative Uncertainty Heatmap

In [ ]:
if HAS_MATPLOTLIB:
    # Create a heatmap of measurement quality
    
    measurements = {
        'Patient A': {
            'Half-life': Knowledge(4.5, 0.3, 'fit_A'),
            'Clearance': Knowledge(14.0, 2.0, 'fit_A'),
            'Bioavailability': Knowledge(0.90, 0.03, 'fit_A'),
        },
        'Patient B': {
            'Half-life': Knowledge(5.2, 0.8, 'fit_B'),
            'Clearance': Knowledge(11.0, 3.0, 'fit_B'),
            'Bioavailability': Knowledge(0.75, 0.08, 'fit_B'),
        },
        'Patient C': {
            'Half-life': Knowledge(4.0, 0.2, 'fit_C'),
            'Clearance': Knowledge(16.0, 1.5, 'fit_C'),
            'Bioavailability': Knowledge(0.95, 0.02, 'fit_C'),
        },
    }
    
    # Extract relative uncertainties
    patients = list(measurements.keys())
    params = ['Half-life', 'Clearance', 'Bioavailability']
    
    rel_unc = np.zeros((len(patients), len(params)))
    for i, patient in enumerate(patients):
        for j, param in enumerate(params):
            rel_unc[i, j] = measurements[patient][param].relative_uncertainty * 100
    
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(rel_unc, cmap='RdYlGn_r', vmin=0, vmax=20)
    
    ax.set_xticks(np.arange(len(params)))
    ax.set_yticks(np.arange(len(patients)))
    ax.set_xticklabels(params)
    ax.set_yticklabels(patients)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
    
    # Add text annotations
    for i in range(len(patients)):
        for j in range(len(params)):
            text = ax.text(j, i, f'{rel_unc[i, j]:.1f}%',
                          ha="center", va="center", color="black", fontsize=10)
    
    ax.set_title('Relative Measurement Uncertainty (%) by Patient', fontsize=14, fontweight='bold')
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Relative Uncertainty (%)', rotation=270, labelpad=20)
    plt.tight_layout()
    plt.show()
else:
    print("# Code to create uncertainty heatmap:")
    print("matplotlib and numpy required for visualization")

## Part 4: Saving Figures for Reports

In [ ]:
# Example: How to save figures for use in reports

if HAS_MATPLOTLIB:
    # Create a simple figure
    fig, ax = plt.subplots(figsize=(8, 5))
    
    data = {
        'Study 1': Knowledge(42.5, 3.2, 'study1'),
        'Study 2': Knowledge(45.0, 2.8, 'study2'),
        'Study 3': Knowledge(40.0, 4.5, 'study3'),
    }
    
    names = list(data.keys())
    values = [data[name].value for name in names]
    errors = [data[name].epsilon for name in names]
    
    x = np.arange(len(names))
    ax.bar(x, values, yerr=errors, capsize=8, alpha=0.7, color='coral')
    ax.set_ylabel('Measurement', fontsize=11)
    ax.set_title('Cross-Study Comparison', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(names)
    ax.grid(axis='y', alpha=0.3)
    
    # Save figure
    fig.savefig('measurement_comparison.png', dpi=300, bbox_inches='tight')
    print("Figure saved as: measurement_comparison.png")
    plt.show()
else:
    print("# To save figures:")
    print("fig.savefig('figure_name.png', dpi=300, bbox_inches='tight')")
    print("fig.savefig('figure_name.pdf', dpi=300, bbox_inches='tight')  # For LaTeX")

## Summary

Key visualization techniques:
1. **Error bars**: Show uncertainty as standard deviation or confidence intervals
2. **Uncertainty bands**: Use `fill_between()` for time-series data
3. **Heatmaps**: Visualize relative uncertainty across conditions
4. **Publication-ready**: Save at high DPI (300+) for print

Next: See notebook 03 for running full pipelines.